In [ ]:
# Portable data paths; execute notebooks from the package root.
import os
from pathlib import Path
_review_base = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()
DATA_ROOT = Path(os.environ.get("REVIEW_DATA_ROOT", str(_review_base / "data"))).resolve()
RESULTS_ROOT = Path(os.environ.get("REVIEW_RESULTS_ROOT", str(_review_base / "results"))).resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


# Energy use per output token

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [ ]:
# Seed for reproducibility
seed = 42
np.random.seed(seed)

# Load model_throughput_consolidated_DB.csv
model_throughput = pd.read_excel(str(DATA_ROOT / '5_Data_Center/TPS_node/TPS_20260821.xlsx'))
print(model_throughput.columns)


In [ ]:
# ---------------------------------------------------------------------
# TPS regression models (unchanged logic from the original script)
# ---------------------------------------------------------------------
def lognorm_params(min_val, max_val):
    """log-normal (mu, sigma) implied by a (5th, 95th) percentile pair."""
    sigma = (np.log(max_val) - np.log(min_val)) / (2 * 1.645)
    mu = np.log(min_val) + 1.645 * sigma
    return mu, sigma
 
 
def create_tps_regression_models(model_data):
    """Per-model log-log regression (or max-TPS fallback) of TPS on (input, output) length."""
    regression_models = {}
    interpolation_models = {}
    max_tps_values = {}
 
    model_data = model_data.copy()
    model_data['TPS_numeric'] = pd.to_numeric(model_data['TPS'], errors='coerce')
    model_data['Input_Length_numeric'] = pd.to_numeric(model_data['Input Length'], errors='coerce')
    model_data['Output_Length_numeric'] = pd.to_numeric(model_data['Output Length'], errors='coerce')
 
    for model_name in model_data['Model'].unique():
        subset = model_data[model_data['Model'] == model_name].dropna(
            subset=['Input_Length_numeric', 'Output_Length_numeric', 'TPS_numeric'])
 
        if len(subset) < 2:
            continue
 
        max_tps_values[model_name] = subset['TPS_numeric'].max()
 
        if len(subset) < 3:
            interpolation_models[model_name] = {
                'type': 'max_tps',
                'max_tps': subset['TPS_numeric'].max(),
                'n_points': len(subset),
            }
        else:
            X = subset[['Input_Length_numeric', 'Output_Length_numeric']].values
            y = subset['TPS_numeric'].values
            X_log = np.log(np.maximum(X, 1e-6))
            y_log = np.log(np.maximum(y, 1e-6))
            reg = LinearRegression().fit(X_log, y_log)
            regression_models[model_name] = {
                'type': 'regression',
                'model': reg,
                'n_points': len(subset),
                'input_range': (X[:, 0].min(), X[:, 0].max()),
                'output_range': (X[:, 1].min(), X[:, 1].max()),
                'max_tps': max_tps_values[model_name],
            }
 
    return regression_models, interpolation_models, max_tps_values
 
 
def predict_tps_batch(model_name, input_lengths, output_lengths,
                       regression_models, interpolation_models, max_tps_values):
    if model_name in regression_models:
        log_features = np.column_stack([
            np.log(np.maximum(input_lengths, 1e-6)),
            np.log(np.maximum(output_lengths, 1e-6)),
        ])
        log_tps_pred = regression_models[model_name]['model'].predict(log_features)
        predicted_tps = np.exp(log_tps_pred)
        return np.minimum(predicted_tps, regression_models[model_name]['max_tps'])
    elif model_name in interpolation_models:
        return np.full(len(input_lengths), interpolation_models[model_name]['max_tps'])
    return np.full(len(input_lengths), np.nan)
 
 
tps_regression_models, interpolation_models, max_tps_values = create_tps_regression_models(model_throughput)
all_tps_models = {**tps_regression_models, **interpolation_models}

In [ ]:
# Build regression models
tps_regression_models, interpolation_models, max_tps_values = create_tps_regression_models(model_throughput)
all_tps_models = {**tps_regression_models, **interpolation_models}


In [ ]:
from sklearn.metrics import r2_score

# tps_regression_models, interpolation_models, max_tps_values = create_tps_regression_models(model_throughput)


print('In-distribution R\xb2 Summary:')
print("=" * 60)

for model_name, info in tps_regression_models.items():
    reg = info['model']

    model_subset = model_throughput[model_throughput['Model'] == model_name].copy()
    model_subset['TPS_numeric'] = pd.to_numeric(model_subset['TPS'], errors='coerce')
    model_subset['Input_Length_numeric'] = pd.to_numeric(model_subset['Input Length'], errors='coerce')
    model_subset['Output_Length_numeric'] = pd.to_numeric(model_subset['Output Length'], errors='coerce')
    model_subset = model_subset.dropna(subset=['Input_Length_numeric', 'Output_Length_numeric', 'TPS_numeric'])

    X = model_subset[['Input_Length_numeric', 'Output_Length_numeric']].values
    y = model_subset['TPS_numeric'].values

    X_log = np.log(np.maximum(X, 1e-6))
    y_log = np.log(np.maximum(y, 1e-6))

    r2_log = reg.score(X_log, y_log)
   

    print(f"{model_name:35s} n={info['n_points']:2d}  R²(log)={r2_log:.4f} ")

# Simulate traditional and test-time scaling outputs

In [ ]:
# ---------------------------------------------------------------------
# Distribution settings -- everything defined by its MEDIAN
# ---------------------------------------------------------------------
n_runs = 100000
TOKENS_PER_UNIT = 1_000_000  # "per million output tokens"
 
 
def exp_scale_from_median(median_tokens):
    """scale (=1/lambda) of an Exponential distribution with the given median."""
    return median_tokens / np.log(2)
 
 
# ---- INPUT token length: median-based (empirical median = 928.00, n=27,303,999) ----
INPUT_LENGTH_MEDIAN = 900
INPUT_LENGTH_SCALE = exp_scale_from_median(INPUT_LENGTH_MEDIAN)
# Note: this matches the empirical MEDIAN, not the empirical MEAN (1631.58).
# A single-parameter exponential can't match both at once (your own KS test
# already flagged the real data isn't a perfect exponential).
 
# ---- OUTPUT token length: two regimes, both median-based ----
OUTPUT_SCENARIOS = {
    'traditional': {
        'label': 'Traditional (conversational, no test-time scaling)',
        'median_tokens': 300,      # reference empirical IQR: 129-618
    },
    'test_time_scaling': {
        'label': 'Test-Time Scaling (reasoning models / long agentic workflows)',
        'median_tokens': 5000,     # reference empirical IQR: 2040-9717
    },
}
for _cfg in OUTPUT_SCENARIOS.values():
    _cfg['scale'] = exp_scale_from_median(_cfg['median_tokens'])
 
 
def get_node_power(model_name):
    return 10.2

pu_range = (0.4, 0.9)          # power utilization, for lognormal
mu_pu, sigma_pu = lognorm_params(*pu_range)


In [ ]:
# ---------------------------------------------------------------------
# Core Monte Carlo simulation -> energy per million OUTPUT tokens
# ---------------------------------------------------------------------
def run_monte_carlo_for_scenario(scenario_key, n_runs=n_runs, seed=seed):

    np.random.seed(seed)
    cfg = OUTPUT_SCENARIOS[scenario_key]
 
    energy_wh_by_model = {}
    energy_per_M_by_model = {}
    tps_by_model = {}
    input_lengths_by_model = {}
    output_lengths_by_model = {}
 
    for model_name in all_tps_models.keys():
        node_power = get_node_power(model_name)
 
        input_lengths = np.round(np.random.exponential(INPUT_LENGTH_SCALE, n_runs)).astype(int)
        input_lengths = np.maximum(input_lengths, 1)
 
        output_lengths = np.round(np.random.exponential(cfg['scale'], n_runs)).astype(int)
        output_lengths = np.maximum(output_lengths, 1)
 
        tokens_per_sec = predict_tps_batch(model_name, input_lengths, output_lengths,
                                            tps_regression_models, interpolation_models, max_tps_values)
 
        valid = tokens_per_sec[~np.isnan(tokens_per_sec)]
        if len(valid) == 0:
            continue
 
        pu = np.random.lognormal(mu_pu, sigma_pu, n_runs)
 
        # Per-query IT-load energy (Wh) -- node power draw only, no PUE multiplier
        energy_kj = node_power * pu * output_lengths / tokens_per_sec
        energy_wh = (energy_kj / 3600) * 1000
 
        # Normalize: energy per million OUTPUT tokens, instead of energy per query
        energy_per_million_tokens = (energy_wh / output_lengths) * TOKENS_PER_UNIT
 
        energy_wh_by_model[model_name] = energy_wh
        energy_per_M_by_model[model_name] = energy_per_million_tokens
        tps_by_model[model_name] = tokens_per_sec
        input_lengths_by_model[model_name] = input_lengths
        output_lengths_by_model[model_name] = output_lengths
 
    return {
        'label': cfg['label'],
        'IT_load_energy_wh_per_query': energy_wh_by_model,
        'IT_load_energy_per_million_output_tokens_Wh': energy_per_M_by_model,
        'tps': tps_by_model,
        'input_lengths': input_lengths_by_model,
        'output_lengths': output_lengths_by_model,
    }
 
 
scenario_results = {key: run_monte_carlo_for_scenario(key) for key in OUTPUT_SCENARIOS}

In [ ]:
# ---------------------------------------------------------------------
# Aggregate metric: ratio-of-sums, NOT mean-of-ratios
# ---------------------------------------------------------------------
# numerator and denominator.
def ratio_of_sums_per_million_tokens(energy_wh, output_lengths):
    return (np.sum(energy_wh) / np.sum(output_lengths)) * TOKENS_PER_UNIT
 
 
def bootstrap_ratio_ci(energy_wh, output_lengths, n_boot=1000, ci=(2.5, 97.5), seed=0):
    rng = np.random.default_rng(seed)
    n = len(energy_wh)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_ratio = (energy_wh[idx].sum(axis=1) / output_lengths[idx].sum(axis=1)) * TOKENS_PER_UNIT
    lo, hi = np.percentile(boot_ratio, ci)
    return lo, hi, boot_ratio.std()
 
 
summary_rows = []
for scenario_key, results in scenario_results.items():
    for model_name in results['IT_load_energy_per_million_output_tokens_Wh'].keys():
        energy_wh = results['IT_load_energy_wh_per_query'][model_name]
        tokens = results['output_lengths'][model_name]
        per_query_ratio = results['IT_load_energy_per_million_output_tokens_Wh'][model_name]
 
        point_estimate = ratio_of_sums_per_million_tokens(energy_wh, tokens)
        ci_lo, ci_hi, boot_std = bootstrap_ratio_ci(energy_wh, tokens)
 
        summary_rows.append({
            'scenario': scenario_key,
            'model': model_name,
            'n': len(energy_wh),
            # correct aggregate ("at-scale") metric
            'IT_load_ratio_of_sums_Wh_per_M_tokens': point_estimate,
            'IT_load_ratio_of_sums_kWh_per_M_tokens': point_estimate / 1000,
            'bootstrap_95CI_lo_Wh_per_M_tokens': ci_lo,
            'bootstrap_95CI_hi_Wh_per_M_tokens': ci_hi,
            # naive mean-of-ratios, kept only for comparison -- NOT the recommended metric
            'mean_of_per_query_ratios_Wh_per_M_tokens': np.mean(per_query_ratio),
            'median_of_per_query_ratios_Wh_per_M_tokens': np.median(per_query_ratio),
        })
 
summary_df = pd.DataFrame(summary_rows).sort_values(
    ['scenario', 'IT_load_ratio_of_sums_Wh_per_M_tokens'], ascending=[True, False])
 
pd.set_option('display.width', 200)
print(summary_df.to_string(index=False))
 
#summary_df.to_csv('data/5_Data_Center/TPS_node/IT_load_energy_per_million_output_tokens_summary.csv', index=False)
#print("\nSaved: IT_load_energy_per_million_output_tokens_summary.csv")

In [ ]:
raw_rows = []
for scenario_key, results in scenario_results.items():
    for model_name, vals in results['IT_load_energy_per_million_output_tokens_Wh'].items():
        raw_rows.append(pd.DataFrame({
            'scenario': scenario_key,
            'model': model_name,
            'energy_per_M_tokens_Wh': vals,
        }))
raw_df = pd.concat(raw_rows, ignore_index=True)
#raw_df.to_excel('data/5_Data_Center/TPS_node/IT_load_energy_per_million_output_tokens_raw.xlsx', index=False)

In [ ]:
col = 'IT_load_ratio_of_sums_kWh_per_M_tokens'
 

traditional_avg = summary_df.loc[summary_df['scenario'] == 'traditional', col].mean()
tts_avg = summary_df.loc[summary_df['scenario'] == 'test_time_scaling', col].mean()
 

WEIGHT_TRADITIONAL = 0.90
WEIGHT_TTS = 0.10
 
weighted_avg = traditional_avg * WEIGHT_TRADITIONAL + tts_avg * WEIGHT_TTS
 
print(f"traditional mean        : {traditional_avg:.6f} kWh/M tokens")
print(f"test_time_scaling mean  : {tts_avg:.6f} kWh/M tokens")
print(f"Weighted mean (90% / 10%)      : {weighted_avg:.6f} kWh/M tokens")

In [ ]:
IT_LOAD_KWH_PER_M_TOKENS_S1 = 0.840728 * 1.10 * 1.10  
IT_LOAD_KWH_PER_M_TOKENS_S2 = 0.840728 * 1.10 * 1.10 * 1.08 
IT_LOAD_KWH_PER_M_TOKENS_S3 = 0.840728 * 1.10 * 1.10 * 1.08 

# Provincial power usage effectiveness

In [ ]:
import geopandas as gpd
gdf = gpd.read_file(str(DATA_ROOT / 'external/BasicData/ChinaMap/\u6709\u5ba1\u56fe\u53f7/2019\u4e2d\u56fd\u5730\u56fe-\u5ba1\u56fe\u53f7GS(2019)1822\u53f7/\u7701_WGS84.shp'))

In [ ]:
import os
import re
import glob
import pandas as pd
 

ROOT_DIR = str(DATA_ROOT / '5_Data_Center/PUE')
OUT_DIR = os.path.join(ROOT_DIR, "Summary")
CASES = range(1, 13)  # case_1 ... case_12
 


# METRIC_PREFIXES = ["PUE", "PUE_single", "WUE", "WUE_single"]
METRIC_PREFIXES = ["PUE", "PUE_single"]
# --------------------------------------------------------------
 
 
def normalize_province_id(raw_id):
    'Normalize province id.'
    try:
        return str(int(float(raw_id)))
    except ValueError:
        
        return raw_id
 
 
def find_metric_files(case_dir, metric_prefix):
    'Find metric files.'
    pattern = os.path.join(case_dir, f"{metric_prefix}_*.csv")
    files = glob.glob(pattern)
    result = {}
    for f in files:
        fname = os.path.basename(f)
        remainder = fname[len(metric_prefix) + 1:]  
        if metric_prefix == "PUE" and (
            remainder.startswith("single_") or remainder.startswith("two_")
        ):
            continue
        raw_id = remainder[:-4]  
        province_id = normalize_province_id(raw_id)
        result[province_id] = f
    return result
 
 
def annual_mean_for_file(filepath):
    'Annual mean for file.'
    df = pd.read_csv(filepath)
    value_cols = [c for c in df.columns if c.lower() != "time"]
    return df[value_cols].mean(axis=0, skipna=True)
 
 
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    long_rows = []
 
    for case_no in CASES:
        case_dir = os.path.join(ROOT_DIR, f"case_{case_no}")
        if not os.path.isdir(case_dir):
            print(f" {case_dir}")
            continue
 
        for metric_prefix in METRIC_PREFIXES:
            files_by_province = find_metric_files(case_dir, metric_prefix)
            if not files_by_province:
                continue
 
            summary_rows = {}
            for province_id, filepath in sorted(files_by_province.items()):
                try:
                    means = annual_mean_for_file(filepath)
                except Exception as e:
                    print(f"[Error] {filepath}: {e}")
                    continue
 
                
                means.index = [f"sample_{i + 1}" for i in range(len(means))]
                summary_rows[province_id] = means
 
                for i, val in enumerate(means.values, start=1):
                    long_rows.append(
                        {
                            "province": province_id,
                            "case": case_no,
                            "metric": metric_prefix,
                            "sample": i,
                            "annual_mean": val,
                        }
                    )
 
            if summary_rows:
                wide_df = pd.DataFrame(summary_rows).T  
                wide_df.index.name = "province"
                out_path = os.path.join(
                    OUT_DIR, f"case_{case_no}_{metric_prefix}_annual_mean.csv"
                )
                wide_df.to_csv(out_path, float_format="%.6f")
                print(
                    f"[Done] case_{case_no} {metric_prefix}: "
                    f"{len(wide_df)} \u4e2a\u7701\u4efd -> {out_path}"
                )
 
    if long_rows:
        long_df = pd.DataFrame(long_rows)
        long_path = os.path.join(OUT_DIR, "1_all_cases_annual_mean_long.csv")
        long_df.to_csv(long_path, index=False, float_format="%.6f")
        print(f" {long_path} ({len(long_df)} ")
    else:
        print('No files matched the configured directories, cases and metric prefixes.')
 
 
if __name__ == "__main__":
    main()

In [ ]:
ROOT_DIR = str(DATA_ROOT / '5_Data_Center/PUE')
OUT_DIR = os.path.join(ROOT_DIR, "Summary")
LONG_FILE = os.path.join(OUT_DIR, "1_all_cases_annual_mean_long.csv")
 

SELECTED_CASES = [1, 2, 3, 4, 5, 6, 7, 11, 12]
# --------------------------------------------------------------

def build_median_table(long_df: pd.DataFrame) -> pd.DataFrame:
    'Build median table.'
    median_df = (
        long_df.groupby(["province", "case"], as_index=False)["annual_mean"]
        .median()
        .rename(columns={"annual_mean": "median_PUE"})
    )
 
    wide = median_df.pivot(index="province", columns="case", values="median_PUE")
    wide.columns = [f"case_{c}" for c in wide.columns]
    wide = wide.reindex(sorted(wide.columns, key=lambda c: int(c.split("_")[1])), axis=1)
    wide.index.name = "province"
    return wide
 
 
def main():
    long_df = pd.read_csv(LONG_FILE)
    long_df["province"] = long_df["province"].astype(str)
 
    wide = build_median_table(long_df)
 
    out_path = os.path.join(OUT_DIR, "province_case_median_PUE.csv")
    wide.to_csv(out_path, float_format="%.6f")
    print(f" {out_path} "
          f"({len(wide)} \u4e2a\u7701\u4efd x {wide.shape[1]} \u4e2acase)")
    print(wide.to_string())
 
    
    cols = [f"case_{c}" for c in SELECTED_CASES]
    subset = wide[cols]
 
    min_max_avg = pd.DataFrame({
        "min_PUE": subset.min(axis=1),
        "max_PUE": subset.max(axis=1),
    })
    min_max_avg["min_max_average_PUE"] = (min_max_avg["min_PUE"] + min_max_avg["max_PUE"]) / 2
 
    min_max_out_path = os.path.join(OUT_DIR, "3_province_min_max_average_PUE.csv")
    min_max_avg.to_csv(min_max_out_path, float_format="%.6f")
    print(f"Saved provincial PUE bounds and midpoints for cases {SELECTED_CASES} to {min_max_out_path}.")
    print(min_max_avg.to_string())
 
 
if __name__ == "__main__":
    main()

# Provincial facility energy per million output tokens

In [ ]:
import geopandas as gpd
gdf = gpd.read_file(str(DATA_ROOT / 'external/BasicData/ChinaMap/\u6709\u5ba1\u56fe\u53f7/2019\u4e2d\u56fd\u5730\u56fe-\u5ba1\u56fe\u53f7GS(2019)1822\u53f7/\u7701_WGS84.shp'))
gdf.head()

In [ ]:
import os
import pandas as pd

# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
ROOT_DIR = str(DATA_ROOT / '5_Data_Center/PUE')
OUT_DIR = os.path.join(ROOT_DIR, "Summary")
PUE_PATH = os.path.join(OUT_DIR, "3_province_min_max_average_PUE.csv")


def normalize_code(x) -> str:
    'Normalize code.'
    return str(int(float(x)))

pue_df = pd.read_csv(PUE_PATH, dtype={'province': str})
pue_df['province'] = pue_df['province'].apply(normalize_code)


code_to_name = {
    normalize_code(code): name
    for code, name in zip(gdf['\u7701\u4ee3\u7801'], gdf['\u7701'])
}
pue_df['province_name'] = pue_df['province'].map(code_to_name)

missing_name = pue_df[pue_df['province_name'].isna()]
if len(missing_name):
    print('See the preceding output for processing details.')
    print(missing_name[['province']].to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------

# Total energy = IT load energy × PUE
# S1: network fabric energy + node redundancy
# S2: network fabric energy + node redundancy + additional idle nodes
# S3: network fabric energy + node redundancy + additional idle nodes
# ---------------------------------------------------------------------

IT_LOAD_KWH_PER_M_TOKENS_S1 = 0.840728 * 1.10 * 1.10
IT_LOAD_KWH_PER_M_TOKENS_S2 = 0.840728 * 1.10 * 1.10 * 1.08
IT_LOAD_KWH_PER_M_TOKENS_S3 = 0.840728 * 1.10 * 1.10 * 1.08

energy_df = pue_df[['province', 'province_name', 'min_PUE', 'max_PUE', 'min_max_average_PUE']].copy()

for s, it_load in [('S1', IT_LOAD_KWH_PER_M_TOKENS_S1), ('S2', IT_LOAD_KWH_PER_M_TOKENS_S2), ('S3', IT_LOAD_KWH_PER_M_TOKENS_S3)]:
    energy_df[f'IT_load_{s}_kWh_per_M_tokens'] = it_load
    energy_df[f'total_energy_{s}_low_kWh_per_M_tokens'] = it_load * energy_df['min_PUE']
    energy_df[f'total_energy_{s}_kWh_per_M_tokens'] = it_load * energy_df['min_max_average_PUE']
    energy_df[f'total_energy_{s}_high_kWh_per_M_tokens'] = it_load * energy_df['max_PUE']

energy_df = energy_df.sort_values('total_energy_S1_kWh_per_M_tokens', ascending=False).reset_index(drop=True)

out_cols = ['province', 'province_name', 'min_PUE', 'max_PUE', 'min_max_average_PUE']
for s in ['S1', 'S2', 'S3']:
    out_cols += [
        f'IT_load_{s}_kWh_per_M_tokens',
        f'total_energy_{s}_low_kWh_per_M_tokens',
        f'total_energy_{s}_kWh_per_M_tokens',
        f'total_energy_{s}_high_kWh_per_M_tokens'
    ]

print(energy_df[out_cols].to_string(index=False))

ENERGY_OUT_PATH = os.path.join(OUT_DIR, "4_province_total_energy_per_million_tokens.csv")
energy_df[out_cols].to_csv(ENERGY_OUT_PATH, index=False, float_format="%.6f")
print(f" {ENERGY_OUT_PATH}")